# WP2 / C2 - main finding (B3): per-stage ROUGE, oracle/search/selection gaps, switching weight, Artikel 149

Runs on its own, Runtime -> Run all (a CPU runtime is fine). It runs the 6 ablation stages (Baseline, +A, +A+E, +A+E+G, +A+E+G+V, +LS) on all 150 articles x seeds 0-29 in parallel, scores every feasible summary once (ROUGE-1/2/L), checks everything against phase B/C/D and C1, and draws 3 figures (Indonesian and English).

Takes ~2-3 h. It's checkpointed per article, so after a disconnect just Run all again. For a quick look set SEEDS = [0] in C0 (~10 min), a full run later only adds the missing seeds. If the parallel pool errors, set WORKERS = 1.

The files I keep afterwards: mowoa_mo_study/wp2/wp2_summary.txt and wp2_per_article.csv. The figures stay on Drive in wp2/figures/.

## C0 - setup

In [ ]:
!pip install pymoo rouge-score -q
import os, json, pickle, time, inspect, collections, platform
import numpy as np, pandas as pd
from multiprocessing import get_context
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy import stats
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/mowoa_mo_study'
PHASEC_DIR = f'{DRIVE_DIR}/phaseC'
PHASED_DIR = f'{DRIVE_DIR}/phaseD'
WP1_DIR    = f'{DRIVE_DIR}/wp1'
WP2_DIR    = f'{DRIVE_DIR}/wp2'
RUN_DIR, E2_DIR, FIG_DIR = f'{WP2_DIR}/ladder_runs', f'{WP2_DIR}/e2_scores', f'{WP2_DIR}/figures'
for d in (WP2_DIR, RUN_DIR, E2_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
CACHE_PATH     = f'{DRIVE_DIR}/article_caches_150.pkl'
CANONICAL_JSON = f'{DRIVE_DIR}/mowoa_summaries_150_rc1_w32_update_test.json'
N_ARTICLES = 150
SEEDS   = list(range(30))      # phase C seeds, [0] for a quick look, a later rerun adds the rest
WORKERS = os.cpu_count() or 1  # articles in parallel (fork), same results per seed either way

## C1 - shared module (same code as phase B/C/D)

In [ ]:
%%writefile mowoa_mo_shared.py
"""
shared layer for the MO-WOA multi-objective comparison (phase B, axis 2).

all four algorithms read the same enumerated objective table and use the same
encoding, repair, initial population per (article, seed) and FE budget, so
objective identity holds by construction.

protocol decisions: D-1 shared P0 (paired design), D-2 terminal + cumulative
fronts, D-3 B = 1480 = 30 + 50 x 29 (surplus goes to the peers, never MO-WOA),
D-4 no tuning, D-6 strata by m = |P*|: A m>=3, B m==2, C m==1 (degenerate).
summary_size=3, repair_solution(), position bonus, knee_weights=[32, 1], the
70/30 leader split and leader preservation are copied verbatim from cell 20 of [v1_4_1].
"""

import numpy as np
from itertools import combinations

# protocol constants (decisions D-1 to D-6 above), don't change them
K_SUMMARY      = 3                       # summary_size (locked)
POP_SIZE       = 30                      # MO-WOA locked
MAX_ITER       = 50                      # MO-WOA locked
FE_BUDGET      = 1480                    # 30 + 50*29   (D-3)
KNEE_WEIGHTS   = np.array([32.0, 1.0])   # w_cov = 32 (locked)
REDUNDANCY_OFFSET = 0.2                  # delta in f2
REF_POINT      = np.array([1.1, 1.1])    # HV reference, normalized space


# per-article artefacts, computed once (BGE-M3 is slow)
class ArticleCache:
    """Everything the optimizer needs for one article. No embeddings at run time."""

    def __init__(self, article_id, adj_matrix, node_importance, position_bonus):
        self.article_id        = article_id
        self.adj               = np.asarray(adj_matrix, dtype=float)
        self.node_importance   = np.asarray(node_importance, dtype=float)
        self.position_bonus    = np.asarray(position_bonus, dtype=float)
        self.n                 = len(self.node_importance)
        # same as cell 20: graph.nodes[i]['importance'] * position_bonus[i]
        self.biased_importance = self.node_importance * self.position_bonus

    def to_dict(self):
        return dict(article_id=self.article_id, adj=self.adj,
                    node_importance=self.node_importance,
                    position_bonus=self.position_bonus)

    @staticmethod
    def from_dict(d):
        return ArticleCache(d['article_id'], d['adj'],
                            d['node_importance'], d['position_bonus'])


def build_position_bonus(n):
    """Copied from cell 28 of [v1_4_1], don't touch."""
    pb = np.ones(n)
    pb[0]   = 3.0
    pb[1:3] *= 2.5
    pb[3:5] *= 1.5
    pb[5:8] *= 0.8
    pb[8:]  *= 0.3
    return pb


# objectives, one implementation for all four algorithms
def evaluate_objectives_reference(solution, cache):
    """
    Literal copy of cell 20's evaluate_objectives(), only kept as ground truth for
    G4 (slow double loop). Returns native (coverage max, redundancy min).
    """
    solution = np.asarray(solution).astype(int)
    selected = np.where(solution == 1)[0]
    coverage = cache.biased_importance[selected].sum() if len(selected) > 0 else 0
    redundancy_raw = 0.0
    num_pairs = 0
    for i_idx, i in enumerate(selected):
        for j_idx, j in enumerate(selected):
            if j_idx > i_idx:
                redundancy_raw += cache.adj[i][j]
                num_pairs += 1
    redundancy = redundancy_raw / max(1, num_pairs) if num_pairs > 0 else 0
    redundancy = max(0, redundancy - REDUNDANCY_OFFSET)
    return coverage, redundancy


def evaluate_objectives_vec(combos, cache):
    """Vectorized: (M, k) sorted combos -> (M, 2) native (coverage, redundancy). Checked by G4."""
    combos = np.asarray(combos, dtype=int)
    M, k = combos.shape
    coverage = cache.biased_importance[combos].sum(axis=1)

    n_pairs = k * (k - 1) // 2
    if n_pairs == 0:
        redundancy = np.zeros(M)
    else:
        iu, ju = np.triu_indices(k, k=1)          # all i<j pairs within a subset
        rows = combos[:, iu]                      # (M, n_pairs)
        cols = combos[:, ju]
        redundancy_raw = cache.adj[rows, cols].sum(axis=1)
        redundancy = redundancy_raw / n_pairs
        redundancy = np.maximum(0.0, redundancy - REDUNDANCY_OFFSET)

    return np.column_stack([coverage, redundancy])


# repair operator, copied from cell 20. don't touch
def repair_solution(solution, cache, target_length=K_SUMMARY):
    solution = np.asarray(solution).astype(int).copy()
    bi = cache.biased_importance
    selected = np.where(solution == 1)[0]
    num_selected = len(selected)

    if num_selected > target_length:
        drop_order = selected[np.argsort(bi[selected])]        # lowest importance first
        for idx in drop_order[:num_selected - target_length]:
            solution[idx] = 0
    elif num_selected < target_length:
        unselected = np.where(solution == 0)[0]
        if len(unselected) > 0:
            add_order = unselected[np.argsort(-bi[unselected])]  # highest importance first
            for idx in add_order[:target_length - num_selected]:
                solution[idx] = 1

    if not solution.any():
        solution[np.argmax(bi)] = 1
    return solution


class Enumeration:
    """All C(N, k) subsets of one article: the exact true front (not a pooled one) + O(1) lookups."""

    def __init__(self, cache, k=K_SUMMARY):
        self.cache = cache
        self.k = k
        n = cache.n
        if n < k:
            raise ValueError(f"article {cache.article_id}: N={n} < k={k}")

        self.combos = np.array(list(combinations(range(n), k)), dtype=np.int16)
        self.M = len(self.combos)
        self.F_native = evaluate_objectives_vec(self.combos, cache)   # (M,2) (cov, red)

        # minimization form used by pymoo and every indicator: (-coverage, redundancy)
        self.F_min = np.column_stack([-self.F_native[:, 0], self.F_native[:, 1]])

        # O(1) lookup:  sorted index tuple -> row in F_native
        self.index_of = {tuple(c): i for i, c in enumerate(map(tuple, self.combos))}

        # normalize on the full feasible set, never on algorithm output
        cov = self.F_native[:, 0]
        red = self.F_native[:, 1]
        self.cov_min, self.cov_max = cov.min(), cov.max()
        self.red_min, self.red_max = red.min(), red.max()
        self.cov_range = self.cov_max - self.cov_min
        self.red_range = self.red_max - self.red_min

        # exact true front, deduped in objective space
        nd = non_dominated_mask(self.F_min)
        F_pf = self.F_min[nd]
        self.P_star_min = unique_rows(F_pf)                 # (m, 2) minimization
        self.P_star_norm = self.normalize(self.P_star_min)
        self.m = len(self.P_star_min)                       # |P*|  -> stratum

        # HV of the true front = max attainable HV
        self.hv_star = hypervolume(self.P_star_norm, REF_POINT)

        # true coverage extreme + true knee, for the hit-rate metrics
        self.true_cov_extreme_idx = int(np.argmax(cov))
        self.true_knee_row = self._true_knee()

    def normalize(self, F_min):
        """Map minimization objectives into per-article [0,1]^2 (both minimized)."""
        F_min = np.atleast_2d(np.asarray(F_min, dtype=float))
        cov = -F_min[:, 0]
        red = F_min[:, 1]
        cr = self.cov_range if self.cov_range > 0 else 1.0
        rr = self.red_range if self.red_range > 0 else 1.0
        g1 = (self.cov_max - cov) / cr          # 0 = best coverage
        g2 = (red - self.red_min) / rr          # 0 = least redundant
        return np.column_stack([g1, g2])

    def F_of(self, solution):
        """O(1) lookup for a binary vector, native (coverage, redundancy)."""
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.F_native[self.index_of[key]]

    def row_of(self, solution):
        key = tuple(np.where(np.asarray(solution).astype(int) == 1)[0].tolist())
        return self.index_of[key]

    def _true_knee(self):
        """Knee of the true front with the locked knee_weights = [32, 1]."""
        P = self.P_star_min
        if len(P) == 1:
            return 0
        native = np.column_stack([-P[:, 0], P[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rng = omax - omin
        rng[rng == 0] = 1
        norm = (native - omin) / rng
        norm[:, 0] = 1 - norm[:, 0]                       # invert coverage
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        return int(d.argmin())

    @property
    def stratum(self):
        if self.m >= 3:
            return 'A'
        elif self.m == 2:
            return 'B'
        return 'C'


# dominance utils, minimization everywhere
def non_dominated_mask(F):
    """Mask of non-dominated rows, F is (n, 2) minimization."""
    F = np.atleast_2d(np.asarray(F, dtype=float))
    n = len(F)
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]:
            continue
        # j dominates i  iff  all(F[j] <= F[i]) and any(F[j] < F[i])
        le = np.all(F <= F[i], axis=1)
        lt = np.any(F < F[i], axis=1)
        if np.any(le & lt):
            keep[i] = False
    return keep


def unique_rows(F, decimals=12):
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    _, idx = np.unique(np.round(F, decimals), axis=0, return_index=True)
    return F[np.sort(idx)]


def extract_front(F):
    """
    Applied the same way to all four. MOEA/D returns its whole population incl.
    dominated members (30 vs NSGA-II's 6), without this it gets scored on junk.
    """
    F = np.atleast_2d(np.asarray(F, dtype=float))
    if len(F) == 0:
        return F
    return unique_rows(F[non_dominated_mask(F)])


# indicators
def hypervolume(F_norm, ref=REF_POINT):
    """Exact 2-D hypervolume by sweep. F_norm: (n,2) minimization, in [0,1]^2."""
    F = np.atleast_2d(np.asarray(F_norm, dtype=float))
    F = F[np.all(F <= ref, axis=1)]           # drop points outside the ref cuboid
    if len(F) == 0:
        return 0.0
    F = unique_rows(F[non_dominated_mask(F)])
    F = F[np.argsort(F[:, 0])]                # ascending f1
    hv, prev_f2 = 0.0, ref[1]
    for f1, f2 in F:
        if f2 < prev_f2:
            hv += (ref[0] - f1) * (prev_f2 - f2)
            prev_f2 = f2
    return float(hv)


def igd(S_norm, P_norm):
    """IGD: mean distance from each true front point to S."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(P[:, None, :] - S[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def igd_plus(S_norm, P_norm):
    """IGD+ (Ishibuchi et al., 2015) -- weakly Pareto-compliant, unlike IGD."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    diff = np.maximum(S[None, :, :] - P[:, None, :], 0.0)   # (|P|, |S|, 2)
    d = np.sqrt((diff ** 2).sum(axis=2))
    return float(d.min(axis=1).mean())


def gd(S_norm, P_norm):
    """GD: mean distance from each point of S to the true front."""
    S = np.atleast_2d(S_norm); P = np.atleast_2d(P_norm)
    if len(S) == 0:
        return float('inf')
    d = np.linalg.norm(S[:, None, :] - P[None, :, :], axis=2)
    return float(d.min(axis=1).mean())


def spacing(S_norm):
    """Schott (1995). Undefined for |S| < 2 -> NaN (frequency of this is reported)."""
    S = np.atleast_2d(S_norm)
    n = len(S)
    if n < 2:
        return float('nan')
    d = np.abs(S[:, None, :] - S[None, :, :]).sum(axis=2)   # L1
    np.fill_diagonal(d, np.inf)
    di = d.min(axis=1)
    return float(np.sqrt(((di.mean() - di) ** 2).sum() / (n - 1)))


def set_coverage(A, B):
    """C(A,B) = fraction of B weakly dominated by A (Zitzler & Thiele 1998). Not symmetric, report both ways."""
    A = np.atleast_2d(A); B = np.atleast_2d(B)
    if len(B) == 0:
        return float('nan')
    if len(A) == 0:
        return 0.0
    covered = 0
    for b in B:
        le = np.all(A <= b, axis=1)
        lt = np.any(A < b, axis=1)
        if np.any(le & lt) or np.any(np.all(A == b, axis=1)):
            covered += 1
    return covered / len(B)


# FE counter, this is what enforces the budget (D-3)
class EvalLog:
    """
    Logs every new candidate evaluated, in order. 1 FE = 1 new candidate.
    Re-evaluating an unchanged individual doesn't count: pymoo never does it, and
    cell 20 recomputes ~3,100 times per run just for logging.
    """

    def __init__(self, enum, budget=FE_BUDGET):
        self.enum = enum
        self.budget = budget
        self.rows = []        # index into enum.F_native, one per new candidate

    def log(self, solution):
        self.rows.append(self.enum.row_of(solution))
        return self.enum.F_native[self.rows[-1]]

    def log_many(self, X):
        for x in X:
            self.log(x)

    @property
    def n_eval(self):
        return len(self.rows)

    def cumulative_front_min(self):
        """Non-dominated subset of the first `budget` evals, so parity is exact."""
        rows = self.rows[:self.budget]
        if not rows:
            return np.empty((0, 2))
        return extract_front(self.enum.F_min[np.array(rows)])


# shared init (D-1), same P0 for all four
def make_initial_population(cache, seed, pop_size=POP_SIZE, target_length=K_SUMMARY):
    """
    cell 20's initialize_population(), but seeded. All four get this P0 for a given
    (article, seed): no init head start, and the paired design makes Wilcoxon signed-rank legit.
    """
    rng = np.random.default_rng(seed)
    n = cache.n
    init_scores = cache.node_importance * cache.position_bonus
    s = init_scores.sum()
    probs = init_scores / s if s > 0 else np.ones(n) / n

    pop = []
    for _ in range(pop_size):
        sol = np.zeros(n, dtype=int)
        num_to_select = max(1, rng.integers(max(1, target_length - 1), target_length + 2))
        chosen = rng.choice(n, size=min(int(num_to_select), n), replace=False, p=probs)
        sol[chosen] = 1
        pop.append(repair_solution(sol, cache, target_length))
    return np.array(pop, dtype=int)


# MO-WOA, cell 20 logic rewired to the shared layer
def _crowding_distance(objectives):
    """Copied from cell 20's crowding_distance()."""
    objectives = np.atleast_2d(objectives)
    n_sol, n_obj = objectives.shape
    if n_sol <= 2:
        return np.full(n_sol, np.inf)
    dist = np.zeros(n_sol)
    for m in range(n_obj):
        order = np.argsort(objectives[:, m])
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng_m = objectives[order[-1], m] - objectives[order[0], m]
        if rng_m == 0:
            continue
        for i in range(1, n_sol - 1):
            dist[order[i]] += (objectives[order[i + 1], m] -
                               objectives[order[i - 1], m]) / rng_m
    return dist


def _pareto_indices(objectives):
    """cell 20's compute_pareto_front in native form (coverage max, redundancy min)."""
    objectives = np.atleast_2d(objectives)
    idx = []
    for i, o1 in enumerate(objectives):
        dominated = False
        for j, o2 in enumerate(objectives):
            if i != j and (o2[0] >= o1[0] and o2[1] <= o1[1] and
                           (o2[0] > o1[0] or o2[1] < o1[1])):
                dominated = True
                break
        if not dominated:
            idx.append(i)
    return idx


def _sigmoid(x):
    """cell 20's sigmoid incl. the [-500, 500] clip. MOPSO uses it too."""
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def run_mowoa(enum, seed, pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET):
    """
    MO-WOA as in cell 20 of [v1_4_1]: binary WOA update, CD leader on the front,
    70/30 leader following, leader kept (so 29 new per gen), repair after every
    update, knee with [32, 1]. FEs: 30 + 50 x 29 = 1480 exactly.
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)                                   # 30 FEs

    gen_history = []          # per-generation population objectives (minimization)

    def objs(pop):            # lookups, not counted as FEs (see EvalLog)
        return np.array([enum.F_of(s) for s in pop])

    def update_position_woa(current, guide, a, pop):
        new_pos = np.zeros(n, dtype=int)
        p = rng.random()
        b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()

        if p < 0.5:
            if abs(A_coeff) < 1:                                # shrinking encircling
                ref = guide
            else:                                               # random search
                ref = pop[rng.integers(len(pop))]
            for i in range(n):
                D = abs(C_coeff * ref[i] - current[i])
                val = ref[i] - A_coeff * D
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:                                                   # spiral bubble-net
            l = rng.uniform(-1, 1)
            for i in range(n):
                Dp = abs(guide[i] - current[i])
                val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0

        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        O = objs(population)
        gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))

        pidx = _pareto_indices(O)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        if len(pidx) > 1:
            cd = _crowding_distance(O[pidx])
            inf_mask = np.isinf(cd)
            if np.any(inf_mask):
                lead_in_pareto = rng.choice(np.where(inf_mask)[0])
            else:
                lead_in_pareto = int(np.argmax(cd))
            leader_idx = pidx[lead_in_pareto]
        else:
            leader_idx = pidx[0]
        leader = population[leader_idx]

        new_pop = []
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)                             # leader kept
            else:
                guide = leader if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = update_position_woa(sol, guide, a, population)
                log.log(child)                                  # 29 FEs per generation
                new_pop.append(child)
        population = np.array(new_pop)

    O = objs(population)
    gen_history.append(np.column_stack([-O[:, 0], O[:, 1]]))
    F_min_terminal = np.column_stack([-O[:, 0], O[:, 1]])

    # knee point, same as cell 20
    pidx = _pareto_indices(O)
    knee_row = None
    if len(pidx) > 0:
        P = O[pidx]
        omin, omax = P.min(axis=0), P.max(axis=0)
        rngv = omax - omin
        rngv[rngv == 0] = 1
        norm = (P - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(population[pidx[int(d.argmin())]])

    return dict(
        name='MO-WOA',
        terminal_front=extract_front(F_min_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=knee_row,
        eval_rows=log.rows,
    )


# binary MOPSO-CD, hand-coded (pymoo's MOPSO is real-valued only)
def run_mopso(enum, seed, pop_size=POP_SIZE, budget=FE_BUDGET,
              w=0.729, c1=1.49445, c2=1.49445, v_max=6.0, archive_size=POP_SIZE):
    """
    Binary MOPSO-CD (Raquel & Naval, 2005). Used instead of the grid MOPSO
    (Coello Coello, Pulido & Lechuga, 2004) because it picks leaders by crowding
    distance like MO-WOA, so the update equation is the only thing that differs:
        particle : w*v + c1*r1*(pbest - x) + c2*r2*(gbest - x)
        whale    : shrinking encircling + spiral bubble-net + random search
    Same sigmoid, repair, init and budget. Clerc constriction defaults, not tuned (D-4).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    B = make_initial_population(cache, seed, pop_size)          # same P0 as MO-WOA (D-1)
    log.log_many(B)                                            # 30 FEs

    # continuous position seeded from the binary start (sigmoid(+/-2) ~ 0.88 / 0.12)
    X = np.where(B == 1, 2.0, -2.0).astype(float)
    V = np.zeros((pop_size, n))

    F = np.array([enum.F_of(b) for b in B])                    # native (cov, red)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])

    pbest_B, pbest_Fmin = B.copy(), F_min.copy()
    arch_B, arch_Fmin = _archive_update(B, F_min, np.empty((0, n), int),
                                        np.empty((0, 2)), archive_size)
    gen_history = [F_min.copy()]

    while log.n_eval < budget:
        # leader: binary tournament on crowding distance
        cd = _crowding_distance(arch_Fmin) if len(arch_Fmin) > 1 else np.array([np.inf])
        for i in range(pop_size):
            if len(arch_B) == 1:
                g = arch_B[0]
            else:
                a1, a2 = rng.integers(len(arch_B), size=2)
                g = arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]

            r1, r2 = rng.random(n), rng.random(n)
            V[i] = (w * V[i]
                    + c1 * r1 * (np.where(pbest_B[i] == 1, 2.0, -2.0) - X[i])
                    + c2 * r2 * (np.where(g == 1, 2.0, -2.0) - X[i]))
            V[i] = np.clip(V[i], -v_max, v_max)
            X[i] = X[i] + V[i]
            b = (rng.random(n) < _sigmoid(X[i])).astype(int)   # same sigmoid as MO-WOA
            B[i] = repair_solution(b, cache)                   # same repair as MO-WOA

        F = np.array([enum.F_of(b) for b in B])
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        log.log_many(B)                                        # 30 new candidates
        gen_history.append(F_min.copy())

        # pbest by dominance, tie -> keep new with prob 0.5
        for i in range(pop_size):
            new, old = F_min[i], pbest_Fmin[i]
            if np.all(new <= old) and np.any(new < old):
                pbest_B[i], pbest_Fmin[i] = B[i].copy(), new
            elif not (np.all(old <= new) and np.any(old < new)):
                if rng.random() < 0.5:
                    pbest_B[i], pbest_Fmin[i] = B[i].copy(), new

        arch_B, arch_Fmin = _archive_update(B, F_min, arch_B, arch_Fmin, archive_size)

    return dict(
        name='MOPSO',
        terminal_front=extract_front(arch_Fmin),               # native output is the archive
        cumulative_front=log.cumulative_front_min(),
        gen_history=gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


def _archive_update(B, F_min, arch_B, arch_Fmin, cap):
    """Merge population into the external archive; prune by crowding distance."""
    allB = np.vstack([arch_B, B]) if len(arch_B) else B.copy()
    allF = np.vstack([arch_Fmin, F_min]) if len(arch_Fmin) else F_min.copy()

    keep = non_dominated_mask(allF)
    allB, allF = allB[keep], allF[keep]

    _, uidx = np.unique(np.round(allF, 12), axis=0, return_index=True)
    allB, allF = allB[np.sort(uidx)], allF[np.sort(uidx)]

    if len(allF) > cap:                                        # prune the most crowded
        cd = _crowding_distance(allF)
        keep_idx = np.argsort(-cd)[:cap]
        allB, allF = allB[keep_idx], allF[keep_idx]
    return allB, allF


# pymoo adapters for NSGA-II and MOEA/D
def _pymoo_imports():
    from pymoo.core.problem import Problem
    from pymoo.core.repair import Repair
    from pymoo.core.sampling import Sampling
    from pymoo.core.callback import Callback
    return Problem, Repair, Sampling, Callback


def make_pymoo_components(enum, seed, budget=FE_BUDGET, pop_size=POP_SIZE):
    Problem, Repair, Sampling, Callback = _pymoo_imports()
    cache = enum.cache
    log = EvalLog(enum, budget)

    class SummaryProblem(Problem):
        def __init__(self):
            # needs vtype=bool
            super().__init__(n_var=cache.n, n_obj=2, xl=0, xu=1, vtype=bool)

        def _evaluate(self, X, out, **kw):
            Xi = np.asarray(X).astype(int)
            F = np.array([enum.F_of(x) for x in Xi])
            log.log_many(Xi)                                   # every row = 1 new FE
            out["F"] = np.column_stack([-F[:, 0], F[:, 1]])    # minimization form

    class SummaryRepair(Repair):
        def _do(self, problem, X, **kw):
            # has to return bool. with int64, BitflipMutation flips via bitwise NOT
            # (~0 = -1, ~1 = -2), the encoding gets corrupted and the run still
            # finishes without error (cardinality sums go negative). hit this in testing
            return np.array([repair_solution(x, cache) for x in np.asarray(X).astype(int)]).astype(bool)

    class SharedSampling(Sampling):
        """D-1: every algorithm gets MO-WOA's importance-biased P0."""
        def _do(self, problem, n_samples, **kw):
            P0 = make_initial_population(cache, seed, pop_size)
            if n_samples <= len(P0):
                return P0[:n_samples].astype(bool)
            extra = make_initial_population(cache, seed + 100000,
                                            n_samples - len(P0))
            return np.vstack([P0, extra]).astype(bool)

    class GenCallback(Callback):
        def __init__(self):
            super().__init__()
            self.gen_history = []
        def notify(self, algorithm):
            self.gen_history.append(algorithm.pop.get("F").copy())

    return SummaryProblem(), SummaryRepair(), SharedSampling(), GenCallback(), log


def run_pymoo(enum, seed, which, budget=FE_BUDGET, pop_size=POP_SIZE):
    """
    which: 'NSGA-II' | 'MOEA/D'
    pymoo only checks termination at generation boundaries, so it can't stop at
    exactly 1480 (NSGA-II with ('n_evals', 1500) used 1524). Peers overshoot to
    the next boundary, so the surplus goes to them, never MO-WOA. The cumulative
    front is cut at the first 1480 evals, so parity is exact there.
    """
    from pymoo.algorithms.moo.nsga2 import NSGA2
    from pymoo.algorithms.moo.moead import MOEAD
    from pymoo.operators.crossover.pntx import TwoPointCrossover
    from pymoo.operators.mutation.bitflip import BitflipMutation
    from pymoo.util.ref_dirs import get_reference_directions
    from pymoo.termination.max_eval import MaximumFunctionCallTermination
    from pymoo.optimize import minimize

    problem, repair, sampling, cb, log = make_pymoo_components(enum, seed, budget, pop_size)

    common = dict(sampling=sampling, crossover=TwoPointCrossover(),
                  mutation=BitflipMutation(), repair=repair)

    if which == 'NSGA-II':
        algo = NSGA2(pop_size=pop_size, eliminate_duplicates=True, **common)
    elif which == 'MOEA/D':
        ref_dirs = get_reference_directions("uniform", 2, n_partitions=pop_size - 1)
        algo = MOEAD(ref_dirs=ref_dirs, n_neighbors=15, **common)
    else:
        raise ValueError(which)

    res = minimize(problem, algo, MaximumFunctionCallTermination(budget),
                   seed=seed, callback=cb, verbose=False, save_history=False)

    # MOEA/D returns its whole population, extract_front cleans that up (same for all four)
    F_terminal = np.atleast_2d(res.pop.get("F"))

    return dict(
        name=which,
        terminal_front=extract_front(F_terminal),
        cumulative_front=log.cumulative_front_min(),
        gen_history=cb.gen_history,
        n_eval=log.n_eval,
        knee_row=None,
        eval_rows=log.rows,
    )


# one (article, seed) -> all four algorithms
ALGORITHMS = ['MO-WOA', 'NSGA-II', 'MOPSO', 'MOEA/D']


def run_all(enum, seed):
    return {
        'MO-WOA':  run_mowoa(enum, seed),
        'NSGA-II': run_pymoo(enum, seed, 'NSGA-II'),
        'MOPSO':   run_mopso(enum, seed),
        'MOEA/D':  run_pymoo(enum, seed, 'MOEA/D'),
    }


def score_run(result, enum, tier='terminal'):
    """Compute every indicator for one run. tier: 'terminal' | 'cumulative' (D-2)."""
    F = result[f'{tier}_front']
    S = enum.normalize(F) if len(F) else np.empty((0, 2))
    P = enum.P_star_norm

    hv = hypervolume(S)
    out = {
        'algorithm': result['name'],
        'tier': tier,
        'HV': hv,
        'HV_pct': 100.0 * hv / enum.hv_star if enum.hv_star > 0 else float('nan'),
        'IGD': igd(S, P),
        'IGD+': igd_plus(S, P),
        'GD': gd(S, P),
        'Spacing': spacing(S),
        'cardinality': len(S),
        'n_eval': result['n_eval'],
    }
    # only possible because we know the true front
    if len(F):
        pf = unique_rows(enum.P_star_min)
        hits = sum(1 for f in F if np.any(np.all(np.isclose(pf, f), axis=1)))
        out['front_recovery'] = hits / len(pf)
        cov_ext = enum.F_min[enum.true_cov_extreme_idx]
        out['found_cov_extreme'] = bool(np.any(np.all(np.isclose(F, cov_ext), axis=1)))
    else:
        out['front_recovery'] = 0.0
        out['found_cov_extreme'] = False
    return out


def hv_convergence(result, enum):
    """HV-vs-generation, normalized as a % of the true optimal HV."""
    return [100.0 * hypervolume(enum.normalize(extract_front(Fg))) / enum.hv_star
            if enum.hv_star > 0 else float('nan')
            for Fg in result['gen_history']]


# MO-WOA+, the fixed variant. canonical MO-WOA above stays untouched.
# pilot diagnosis (real data):
#   retention: terminal 75.70% vs cumulative 87.28% HV, 11.58 pts thrown away.
#     no archive, generational replacement drops what it found
#   search: S-transfer saturates. as a->0, A->0, val -> leader[i] in {0,1},
#     sigmoid(val) -> {0.50, 0.73}, so every bit is a near coin-flip and
#     repair_solution() collapses the offspring onto greedy top-3. diversity dies,
#     search only adds ~4.6 pts in 1480 FEs
# fixes, each switchable for the ablation:
#   use_archive: external archive, cap 30 like MOPSO, non-dominated, crowding-pruned
#     [Coello Coello et al., 2004]
#   transfer='V': V-shaped transfer [Mirjalili & Lewis, 2013; binary WOA: Hussien
#     et al., 2019]. step magnitude -> flip prob of the parent's bit, step->0 keeps the parent

def _v_transfer(step):
    """V-shaped transfer: |tanh(step)|. V(0)=0 (keep parent bit), V(large)->1."""
    return np.abs(np.tanh(step))


def _nds_crowding_select(B_all, F_all, k):
    """(mu+lambda) environmental selection: non-dominated sorting + crowding (NSGA-II)."""
    Nn = len(F_all)
    remaining = np.arange(Nn)
    chosen = []
    while len(chosen) < k and len(remaining) > 0:
        mask = non_dominated_mask(F_all[remaining])
        front = remaining[mask]
        if len(chosen) + len(front) <= k:
            chosen.extend(front.tolist())
            remaining = remaining[~mask]
        else:
            cd = _crowding_distance(F_all[front])
            need = k - len(chosen)
            chosen.extend(front[np.argsort(-cd)[:need]].tolist())
            break
    chosen = np.array(chosen[:k])
    return B_all[chosen], F_all[chosen]


def run_mowoa_final(enum, seed, transfer='V', guide_mode='per_whale_archive',
                    survival='elitist', use_archive=True,
                    ls_probes=0, ls_mode='directed',
                    pop_size=POP_SIZE, max_iter=MAX_ITER, budget=FE_BUDGET,
                    archive_size=POP_SIZE):
    """
    Ablation-switchable MO-WOA+. Update equations, repair, init (D-1), budget (D-3)
    and knee weights are the same as cell 20. Switches:
      transfer   : 'S' (cell 20 sigmoid) | 'V' (V-shaped flip)
      guide_mode : 'single_leader' (cell 20) | 'per_whale_archive' (MOPSO-CD gbest rule)
      survival   : 'generational' (cell 20) | 'elitist' (mu+lambda NDS+crowding, like NSWOA)
      use_archive: external archive, cap 30, crowding-pruned
    S / single_leader / generational / no archive reproduces run_mowoa() bit-exactly (G7b).
    """
    cache = enum.cache
    rng = np.random.default_rng(seed)
    n = cache.n
    log = EvalLog(enum, budget)

    population = make_initial_population(cache, seed, pop_size)
    log.log_many(population)

    def objs(pop):
        return np.array([enum.F_of(s) for s in pop])

    F = objs(population)
    F_min = np.column_stack([-F[:, 0], F[:, 1]])
    gen_history = [F_min.copy()]

    arch_B = np.empty((0, n), dtype=int); arch_F = np.empty((0, 2))
    if use_archive:
        arch_B, arch_F = _archive_update(population, F_min, arch_B, arch_F, archive_size)

    def offspring_of(current, guide, a, pop):
        p = rng.random(); b = 1.0
        A_coeff = 2 * a * rng.random() - a
        C_coeff = 2 * rng.random()
        if transfer == 'S':
            new_pos = np.zeros(n, dtype=int)
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    val = ref[i] - A_coeff * D
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    val = Dp * np.exp(b * l) * np.cos(2 * np.pi * l) + guide[i]
                    new_pos[i] = 1 if rng.random() < _sigmoid(val) else 0
        else:
            new_pos = current.copy()
            if p < 0.5:
                ref = guide if abs(A_coeff) < 1 else pop[rng.integers(len(pop))]
                for i in range(n):
                    D = abs(C_coeff * ref[i] - current[i])
                    if rng.random() < _v_transfer(A_coeff * D):
                        new_pos[i] = 1 - new_pos[i]
            else:
                l = rng.uniform(-1, 1)
                for i in range(n):
                    Dp = abs(guide[i] - current[i])
                    if rng.random() < _v_transfer(Dp * np.exp(b * l) * np.cos(2 * np.pi * l)):
                        new_pos[i] = 1 - new_pos[i]
        if not new_pos.any():
            new_pos[rng.integers(n)] = 1
        return repair_solution(new_pos, cache)

    def ls_probe():
        """1-swap probe on an archive member. Cardinality stays at k so repair never
        fires, it's the only move that skips repair's importance bias.
        random: uniform swap. directed: drop the selected sentence most similar to
        the others, add one of the 3 least similar unselected ones."""
        if len(arch_B) == 0:
            return None
        if len(arch_B) == 1:
            base = arch_B[0]
        else:
            cdp = _crowding_distance(arch_F)
            a1, a2 = rng.integers(len(arch_B), size=2)
            base = arch_B[a1] if cdp[a1] >= cdp[a2] else arch_B[a2]
        x = np.asarray(base).astype(int).copy()
        sel = np.where(x == 1)[0]
        unsel = np.where(x == 0)[0]
        if len(sel) == 0 or len(unsel) == 0:
            return None
        if ls_mode == 'random':
            out_i = rng.choice(sel)
            in_i = rng.choice(unsel)
        else:
            contrib = np.array([cache.adj[i, sel].sum() for i in sel])
            out_i = sel[int(np.argmax(contrib))] if rng.random() < 0.7 else rng.choice(sel)
            remaining = sel[sel != out_i]
            scores = cache.adj[unsel][:, remaining].sum(axis=1)
            k = min(3, len(unsel))
            in_i = unsel[rng.choice(np.argsort(scores)[:k])]
        x[out_i] = 0
        x[in_i] = 1
        return x

    for it in range(max_iter):
        a = 2.0 - it * (2.0 / max_iter)
        F = objs(population)
        F_min = np.column_stack([-F[:, 0], F[:, 1]])
        pidx = _pareto_indices(F)
        if len(pidx) == 0:
            pidx = list(range(len(population)))

        # leader / guides
        if guide_mode == 'single_leader':
            if len(pidx) > 1:
                cd = _crowding_distance(F[pidx])
                inf_mask = np.isinf(cd)
                lp = (rng.choice(np.where(inf_mask)[0]) if np.any(inf_mask)
                      else int(np.argmax(cd)))
                leader_idx = pidx[lp]
            else:
                leader_idx = pidx[0]
            leader = population[leader_idx]
        else:
            leader_idx = 0
            leader = None

        def draw_guide():
            if guide_mode == 'per_whale_archive' and use_archive and len(arch_B) > 0:
                if len(arch_B) == 1:
                    return arch_B[0]
                cd = _crowding_distance(arch_F)
                a1, a2 = rng.integers(len(arch_B), size=2)
                return arch_B[a1] if cd[a1] >= cd[a2] else arch_B[a2]
            if guide_mode == 'per_whale_archive':          # archive off: pop front
                if len(pidx) == 1:
                    return population[pidx[0]]
                cd = _crowding_distance(F[pidx])
                a1, a2 = rng.integers(len(pidx), size=2)
                return population[pidx[a1] if cd[a1] >= cd[a2] else pidx[a2]]
            return leader

        # 29 offspring, leader stays in its slot like cell 20 (needed for the
        # bit-exact rng stream at baseline switches)
        new_pop = []
        children = []
        if ls_probes > 0 and use_archive and len(arch_B) > 0:
            nonleader = [i for i in range(len(population)) if i != leader_idx]
            probe_slots = set(rng.choice(nonleader,
                                         size=min(ls_probes, len(nonleader)),
                                         replace=False).tolist())
        else:
            probe_slots = set()
        for i, sol in enumerate(population):
            if i == leader_idx:
                new_pop.append(sol)
                continue
            child = None
            if i in probe_slots:
                child = ls_probe()
            if child is None:
                g = draw_guide() if rng.random() < 0.7 else population[rng.choice(pidx)]
                child = offspring_of(sol, g, a, population)
            log.log(child)
            new_pop.append(child)
            children.append(child)
        children = np.array(children)
        Fc = objs(children)
        Fc_min = np.column_stack([-Fc[:, 0], Fc[:, 1]])

        # survival
        if survival == 'generational':
            population = np.array(new_pop)
        else:                                              # elitist (mu+lambda)
            B_all = np.vstack([population, children])
            F_all = np.vstack([F_min, Fc_min])
            population, _ = _nds_crowding_select(B_all, F_all, pop_size)

        Fq = objs(population)
        Fq_min = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        gen_history.append(Fq_min.copy())
        if use_archive:
            arch_B, arch_F = _archive_update(children, Fc_min, arch_B, arch_F,
                                             archive_size)

    # terminal output + knee
    if use_archive:
        F_terminal = arch_F; knee_B, knee_F = arch_B, arch_F
    else:
        Fq = objs(population)
        F_terminal = np.column_stack([-Fq[:, 0], Fq[:, 1]])
        pidx = _pareto_indices(Fq)
        knee_B, knee_F = population[pidx], F_terminal[pidx]

    knee_row = None
    if len(knee_F) > 0:
        native = np.column_stack([-knee_F[:, 0], knee_F[:, 1]])
        omin, omax = native.min(axis=0), native.max(axis=0)
        rngv = omax - omin; rngv[rngv == 0] = 1
        norm = (native - omin) / rngv
        norm[:, 0] = 1 - norm[:, 0]
        d = np.sqrt((KNEE_WEIGHTS * norm ** 2).sum(axis=1))
        knee_row = enum.row_of(knee_B[int(d.argmin())])

    return dict(name=f"MO-WOA+[{transfer},{guide_mode[:8]},{survival[:4]}"
                     f"{',arch' if use_archive else ''}]",
                terminal_front=extract_front(F_terminal),
                cumulative_front=log.cumulative_front_min(),
                gen_history=gen_history, n_eval=log.n_eval,
                knee_row=knee_row, eval_rows=log.rows)

## C2 - load data, rebuild enumerations, ladder definition

In [ ]:
import importlib, mowoa_mo_shared as mo
importlib.reload(mo)

LOG = []
def out(*a):
    s = ' '.join(str(x) for x in a)
    print(s)
    LOG.append(s)

with open(CACHE_PATH, 'rb') as f:
    caches = [mo.ArticleCache.from_dict(d) for d in pickle.load(f)]
enums = {c.article_id: mo.Enumeration(c) for c in caches}
AIDS = sorted(enums)
HARD = [a for a in AIDS if enums[a].M > mo.FE_BUDGET]
with open(f'{PHASEC_DIR}/sentences.pkl', 'rb') as f:
    SENTS = pickle.load(f)
with open(CANONICAL_JSON, encoding='utf-8') as f:
    golden = json.load(f)['golden_summaries'][:N_ARTICLES]
out('=' * 78)
out('WP2 / C2 run —', time.strftime('%Y-%m-%d %H:%M'), '| python', platform.python_version(),
    '| CPUs', os.cpu_count(), '| workers', WORKERS)
out(f'articles {len(AIDS)} (hard {len(HARD)}) | seeds {SEEDS[0]}..{SEEDS[-1]} ({len(SEEDS)})')

# run_mowoa_final with one change: it also returns the final output set (knee_B, knee_F),
# so every selection weight can be applied to the same archive. search not touched
_src = inspect.getsource(mo.run_mowoa_final)
_src = _src.replace('def run_mowoa_final(', 'def run_mowoa_final_arch(', 1)
_old = 'knee_row=knee_row, eval_rows=log.rows)'
assert _src.count(_old) == 1
_src = _src.replace(_old, 'knee_row=knee_row, eval_rows=log.rows, '
                          'knee_B=np.asarray(knee_B), knee_F=np.asarray(knee_F))')
exec(compile(_src, 'run_mowoa_final_arch', 'exec'), mo.__dict__)
run_arch = mo.run_mowoa_final_arch

BASE = dict(transfer='S', guide_mode='single_leader', survival='generational', use_archive=False)
STAGES = [                                     # = Phase C 'MO-WOA', D11 ladder, Phase C 'MO-WOA+'
    ('Baseline', BASE),
    ('+A',       dict(BASE, use_archive=True)),
    ('+A+E',     dict(BASE, use_archive=True, survival='elitist')),
    ('+A+E+G',   dict(BASE, use_archive=True, survival='elitist', guide_mode='per_whale_archive')),
    ('+A+E+G+V', dict(transfer='V', guide_mode='per_whale_archive', survival='elitist', use_archive=True)),
    ('+LS',      dict(transfer='V', guide_mode='per_whale_archive', survival='elitist', use_archive=True,
                      ls_probes=10, ls_mode='directed')),
]
STAGE_KW = dict(STAGES)
SNAMES = [s for s, _ in STAGES]
FULL = '+LS'
WEIGHTS = {'w1': 1.0, 'w2': 2.0, 'w4': 4.0, 'w8': 8.0, 'w16': 16.0, 'w32': 32.0, 'w64': 64.0}

def _norm(knee_F):
    native = np.column_stack([-knee_F[:, 0], knee_F[:, 1]])
    omin, omax = native.min(axis=0), native.max(axis=0)
    rngv = omax - omin
    rngv[rngv == 0] = 1
    nrm = (native - omin) / rngv
    nrm[:, 0] = 1 - nrm[:, 0]
    return native, nrm

def select_row(knee_B, knee_F, enum, wc):
    """Selection rule of run_mowoa_final / cell 21 with weights (wc, 1)."""
    _, nrm = _norm(knee_F)
    d = np.sqrt((np.array([wc, 1.0]) * nrm ** 2).sum(axis=1))
    return enum.row_of(knee_B[int(d.argmin())])

def switch_weight(knee_F):
    """w*: for w_cov > w*, the rule picks the output set's max-coverage member.
    Member j beats that member iff w*n_cov,j^2 + n_red,j^2 < n_red,ext^2."""
    native, nrm = _norm(knee_F)
    ext = int(np.argmax(native[:, 0]))
    ncov, nred = nrm[:, 0], nrm[:, 1]
    j = np.where((np.arange(len(nrm)) != ext) & (ncov > 0))[0]
    if len(j) == 0:
        return 0.0
    return float(max(0.0, ((nred[ext] ** 2 - nred[j] ** 2) / ncov[j] ** 2).max()))

## C3 - E2 extended: ROUGE-1/2/L and word count of every feasible subset

In [ ]:
from rouge_score import rouge_scorer
SC = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)   # locked

def e2_article(aid):
    e, sents, gold = enums[aid], SENTS[aid], golden[aid]
    wc = np.array([len(s.split()) for s in sents])
    R = np.zeros((e.M, 3), dtype=np.float32)
    for k, combo in enumerate(e.combos):            # rows sorted ascending = document order
        s = SC.score(gold, ' '.join(sents[i] for i in combo))
        R[k] = (s['rouge1'].fmeasure, s['rouge2'].fmeasure, s['rougeL'].fmeasure)
    return aid, R, wc[e.combos.astype(int)].sum(axis=1).astype(np.int16)

def run_parallel(fn, jobs):
    if WORKERS <= 1:
        for j in jobs:
            yield fn(*j)
        return
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=get_context('fork')) as ex:
        futs = [ex.submit(fn, *j) for j in jobs]
        for f in as_completed(futs):
            yield f.result()

t0 = time.time()
todo = [a for a in AIDS if not os.path.exists(f'{E2_DIR}/article_{a:03d}.npz')]
todo.sort(key=lambda a: -enums[a].M)
for k, (aid, R_, W_) in enumerate(run_parallel(e2_article, [(a,) for a in todo])):
    np.savez_compressed(f'{E2_DIR}/article_{aid:03d}.npz', R=R_, words=W_)   # written by the parent
    if k % 25 == 0:
        print(f'  E2 {k + 1}/{len(todo)} articles scored ({time.time() - t0:.0f}s)')
SCORES = {}
for aid in AIDS:
    z = np.load(f'{E2_DIR}/article_{aid:03d}.npz')
    SCORES[aid] = (z['R'].astype(float), z['words'].astype(int))
out(f'\n[C3] E2 scores ready: {sum(enums[a].M for a in AIDS):,} subsets scored '
    f'({len(todo)} new articles, {time.time() - t0:.0f}s)')

PF = {}                                              # rows whose objective vector is on P*
for aid in AIDS:
    e = enums[aid]
    m = np.zeros(e.M, dtype=bool)
    for p in e.P_star_min:
        m |= np.isclose(e.F_min, p).all(axis=1)
    PF[aid] = m
LEAD = {a: enums[a].index_of[(0, 1, 2)] for a in AIDS}
EXT = {a: enums[a].true_cov_extreme_idx for a in AIDS}

## C4 - ladder runs: 6 stages x 150 articles x seeds (checkpointed, parallel)

In [ ]:
def work_article(aid, todo):
    e = enums[aid]
    ext_F = e.F_min[e.true_cov_extreme_idx]
    recs = []
    for stage, seed in todo:
        t0 = time.time()
        r = run_arch(e, seed, **STAGE_KW[stage])
        assert r['n_eval'] == mo.FE_BUDGET
        st = mo.score_run(r, e, 'terminal')
        kB, kF = r['knee_B'], r['knee_F']
        fg = next((g for g, Fg in enumerate(r['gen_history'])
                   if np.any(np.all(np.isclose(Fg, ext_F), axis=1))), -1)
        rec = dict(article=aid, stage=stage, seed=seed, sec=time.time() - t0,
                   n_unique=len(set(r['eval_rows'])), hv=st['HV_pct'], igd=st['IGD'],
                   recov=st['front_recovery'], ext_term=bool(st['found_cov_extreme']),
                   first_gen=fg, out_rows=[e.row_of(b) for b in kB], wstar=switch_weight(kF))
        for name, wc in WEIGHTS.items():
            rec[f'row_{name}'] = select_row(kB, kF, e, wc)
        recs.append(rec)
    return aid, recs

def load_ck(aid):
    ck = f'{RUN_DIR}/article_{aid:03d}.pkl'
    if not os.path.exists(ck):
        return []
    with open(ck, 'rb') as f:
        return pickle.load(f)

jobs = []
for aid in AIDS:
    have = {(r['stage'], r['seed']) for r in load_ck(aid)}
    todo = [(s, sd) for s in SNAMES for sd in SEEDS if (s, sd) not in have]
    if todo:
        jobs.append((aid, todo))
jobs.sort(key=lambda j: -enums[j[0]].cache.n * len(j[1]))
n_runs = sum(len(t) for _, t in jobs)
print(f'ladder: {n_runs} runs to do over {len(jobs)} articles with {WORKERS} worker(s)')
t0, done = time.time(), 0
for aid, recs in run_parallel(work_article, jobs):
    old = load_ck(aid)
    with open(f'{RUN_DIR}/article_{aid:03d}.pkl', 'wb') as f:
        pickle.dump(old + recs, f)
    done += len(recs)
    el = time.time() - t0
    print(f'  article {aid:>3} saved | {done}/{n_runs} runs | {el / 60:.1f} min | '
          f'ETA {el / done * (n_runs - done) / 60:.0f} min')

RUNS = pd.DataFrame([r for aid in AIDS for r in load_ck(aid)])
RUNS = RUNS[RUNS.seed.isin(SEEDS)].reset_index(drop=True)
assert len(RUNS) == len(AIDS) * len(SNAMES) * len(SEEDS), 'missing runs — rerun this cell'
out(f'\n[C4] ladder runs: {len(RUNS)} ({len(SNAMES)} stages x {len(AIDS)} articles x {len(SEEDS)} seeds); '
    f'mean {RUNS.sec.mean():.2f}s/run')

## C5 - reproduction checks (stop if any fails)

In [ ]:
out('\nREPRODUCTION CHECKS')
# E2 R-1 vs phase B cell 14
with open(f'{DRIVE_DIR}/e2_decomposition.pkl', 'rb') as f:
    E2B = {d['article']: d for d in pickle.load(f)}
ok = [np.isclose(SCORES[a][0][:, 0].max(), E2B[a]['oracle'], atol=1e-6)
      and np.isclose(SCORES[a][0][PF[a], 0].max(), E2B[a]['best_on_front'], atol=1e-6)
      and np.isclose(SCORES[a][0][LEAD[a], 0], E2B[a]['greedy'], atol=1e-6) for a in AIDS]
out(f'  E2 R-1 oracle / best-on-front / greedy == Phase B Cell 14: {sum(ok)}/{len(AIDS)}')
# LEAD-3 vs C1
c1 = pd.read_csv(f'{WP1_DIR}/wp1_per_article.csv').set_index('article')
ok = [abs(SCORES[a][0][LEAD[a], 0] - c1.loc[a + 1, 'lead3_r1']) < 6e-5 for a in AIDS]
out(f'  LEAD-3 R-1 == C1 (wp1_per_article.csv): {sum(ok)}/{len(AIDS)}')
# full stage vs the C1 reruns (selection at w=1 and w=32, per run)
c1runs = {}
for a in AIDS:
    with open(f'{WP1_DIR}/mowoa_plus_runs/article_{a:03d}.pkl', 'rb') as f:
        for r in pickle.load(f):
            c1runs[(a, r['seed'])] = (r['row_w1'], r['row_w32'])
F_ = RUNS[RUNS.stage == FULL]
ok = [c1runs.get((r.article, r.seed)) == (r.row_w1, r.row_w32) for r in F_.itertuples()]
out(f'  {FULL} selections (w=1, w=32) == C1 runs: {sum(ok)}/{len(F_)}')
# phase C: knee rows (w=32) and terminal HV for Baseline and +LS
pc = {}
for a in AIDS:
    with open(f'{PHASEC_DIR}/article_{a:03d}.pkl', 'rb') as f:
        for r in pickle.load(f):
            if r['algorithm'] in ('MO-WOA', 'MO-WOA+'):
                pc[(a, r['algorithm'], r['seed'])] = (r['knee_row'], r['term_HV_pct'])
for stage, algo in (('Baseline', 'MO-WOA'), (FULL, 'MO-WOA+')):
    g = RUNS[RUNS.stage == stage]
    k_ok = [pc[(r.article, algo, r.seed)][0] == r.row_w32 for r in g.itertuples()]
    h_ok = [np.isclose(pc[(r.article, algo, r.seed)][1], r.hv) for r in g.itertuples()]
    out(f'  {stage:<9} == Phase C {algo:<8}: w=32 selection {sum(k_ok)}/{len(g)}, terminal HV {sum(h_ok)}/{len(g)}')
# D11 ladder HV on the hard articles
dl = {}
for a in HARD:
    p = f'{PHASED_DIR}/ladder/article_{a:03d}.pkl'
    if os.path.exists(p):
        with open(p, 'rb') as f:
            for r in pickle.load(f):
                dl[(a, r['variant'], r['seed'])] = r['hv']
for stage in ('+A', '+A+E', '+A+E+G', '+A+E+G+V'):
    g = RUNS[(RUNS.stage == stage) & RUNS.article.isin(HARD)]
    h_ok = [np.isclose(dl.get((r.article, stage, r.seed), np.nan), r.hv) for r in g.itertuples()]
    out(f'  {stage:<9} == D11 ladder terminal HV (hard): {sum(h_ok)}/{len(g)}')

## C6 - per-run ROUGE lookups, per-article seed means

In [ ]:
def jacc(a, b, aid):
    A, B = set(enums[aid].combos[a].tolist()), set(enums[aid].combos[b].tolist())
    return len(A & B) / len(A | B)

full_sel = {(r.article, r.seed): (r.row_w1, r.row_w32) for r in F_.itertuples()}
cols = collections.defaultdict(list)
for r in RUNS.itertuples():
    R, W = SCORES[r.article]
    outR = R[r.out_rows]
    for wk in ('w1', 'w32'):
        row = getattr(r, f'row_{wk}')
        for j, m in enumerate(('r1', 'r2', 'rl')):
            cols[f'{m}_{wk}'].append(R[row, j])
        cols[f'wd_{wk}'].append(W[row])
        cols[f'lead_{wk}'].append(row == LEAD[r.article])
        cols[f'onpf_{wk}'].append(bool(PF[r.article][row]))
        cols[f'jac_{wk}'].append(jacc(row, full_sel[(r.article, r.seed)][0 if wk == 'w1' else 1], r.article))
        cols[f'isbest_{wk}'].append(R[row, 0] >= outR[:, 0].max() - 1e-12)
    for j, m in enumerate(('r1', 'r2', 'rl')):
        cols[f'bestout_{m}'].append(outR[:, j].max())
for k, v in cols.items():
    RUNS[k] = v

ART = pd.DataFrame({'article': AIDS})
ART['N'] = [enums[a].cache.n for a in AIDS]
ART['hard'] = [a in HARD for a in AIDS]
for j, m in enumerate(('r1', 'r2', 'rl')):
    ART[f'oracle_{m}'] = [SCORES[a][0][:, j].max() for a in AIDS]
    ART[f'front_{m}'] = [SCORES[a][0][PF[a], j].max() for a in AIDS]
    ART[f'lead3_{m}'] = [SCORES[a][0][LEAD[a], j] for a in AIDS]
ART['oracle_on_front'] = [bool(PF[a][int(np.argmax(SCORES[a][0][:, 0]))]) for a in AIDS]
ART = ART.set_index('article')
SM = RUNS.groupby(['stage', 'article']).mean(numeric_only=True)     # seed means per article

def sm(stage, col):
    return SM.loc[stage][col].reindex(AIDS).to_numpy()

def paired(x, y):
    d = np.asarray(x) - np.asarray(y); n = len(d)
    m, se = d.mean(), d.std(ddof=1) / np.sqrt(n)
    t = stats.t.ppf(0.975, n - 1)
    return m, m - t * se, m + t * se, int((d > 1e-9).sum()), int((abs(d) <= 1e-9).sum()), int((d < -1e-9).sum())

def ci(x):
    x = np.asarray(x); n = len(x)
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean(), x.mean() - h, x.mean() + h

## C7 - results: Q28 per-stage ROUGE, Q13/Q27 gaps, Q22 switching weight

In [ ]:
P0_ext = []
for a in AIDS:
    for sd in SEEDS:
        P0 = mo.make_initial_population(enums[a].cache, sd)
        P0_ext.append(any(enums[a].row_of(x) == EXT[a] for x in P0))
out(f'\nCOVERAGE EXTREME (= LEAD-3) IN THE SHARED INITIAL POPULATION P0: {sum(P0_ext)}/{len(P0_ext)} '
    f'(article, seed) pairs')
fg = RUNS[RUNS.first_gen > 0]
out(f'  runs where it first appears after generation 0: {len(fg)} '
    f'(stages: {dict(collections.Counter(fg.stage))})')

out('\nTABEL — PER-STAGE OPTIMIZER QUALITY AND DELIVERED SUMMARY (Q28)')
out('HV% and recovery: mean over 33 hard x seeds runs (= D11 / Tabel 4.15 basis). '
    'ROUGE: per-article seed mean, then mean over 150 articles [95% interval of the mean]')
out('stage\tHV%hard\trecov%hard\tHV%150\tcov_extreme_in_output%\tR1(1,1)\tR2(1,1)\tRL(1,1)\twords(1,1)\t=LEAD3%(1,1)\t'
    'Jacc_vs_full(1,1)\tR1(32,1)\t=LEAD3%(32,1)\tunique_cand')
for s in SNAMES:
    g = RUNS[RUNS.stage == s]; gh = g[g.article.isin(HARD)]
    r1 = ci(sm(s, 'r1_w1'))
    out(f'{s}\t{gh.hv.mean():.2f}±{gh.hv.std():.2f}\t{100 * gh.recov.mean():.1f}\t{g.hv.mean():.2f}\t'
        f'{100 * g.ext_term.mean():.1f}\t{r1[0]:.4f} [{r1[1]:.4f},{r1[2]:.4f}]\t{sm(s, "r2_w1").mean():.4f}\t'
        f'{sm(s, "rl_w1").mean():.4f}\t{sm(s, "wd_w1").mean():.1f}\t{100 * g.lead_w1.mean():.1f}\t'
        f'{g.jac_w1.mean():.3f}\t{sm(s, "r1_w32").mean():.4f}\t{100 * g.lead_w32.mean():.1f}\t'
        f'{g.n_unique.mean():.1f}')
out('\nPAIRED DIFFERENCES vs the full configuration (+LS), per article (seed means), n = 150')
for wk, lab in (('w1', '(1,1)'), ('w32', '(32,1)')):
    for s in SNAMES[:-1]:
        m, lo, hi, h, e_, l = paired(sm(s, f'r1_{wk}'), sm(FULL, f'r1_{wk}'))
        out(f'  {lab:<7} R-1 {s:<9} − +LS: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]  higher/equal/lower {h}/{e_}/{l}')

out('\nDECOMPOSITION (Q13/Q27), ROUGE-1, mean over 150 articles; per-article seed means')
out('oracle − best on P* = objective gap; best on P* − best in output set = search gap; '
    'best in output set − delivered = selection gap')
out('stage\toracle\tbest_P*\tbest_out\tdeliv(1,1)\tdeliv(32,1)\tobj_gap\tsearch_gap\tsel_gap(1,1)\tsel_gap(32,1)\t'
    'deliv_on_P*%(1,1)\tdeliv=best_out%(1,1)')
for s in SNAMES:
    bo, d1, d32 = sm(s, 'bestout_r1'), sm(s, 'r1_w1'), sm(s, 'r1_w32')
    g = RUNS[RUNS.stage == s]
    out(f'{s}\t{ART.oracle_r1.mean():.4f}\t{ART.front_r1.mean():.4f}\t{bo.mean():.4f}\t{d1.mean():.4f}\t'
        f'{d32.mean():.4f}\t{(ART.oracle_r1 - ART.front_r1).mean():.4f}\t{(ART.front_r1.to_numpy() - bo).mean():+.4f}\t'
        f'{(bo - d1).mean():.4f}\t{(bo - d32).mean():.4f}\t{100 * g.onpf_w1.mean():.1f}\t{100 * g.isbest_w1.mean():.1f}')
out(f'oracle on P* (R-1): {int(ART.oracle_on_front.sum())}/150 ({100 * ART.oracle_on_front.mean():.1f}%)')
out('\nGAP DISTRIBUTIONS, full configuration (+LS), ROUGE-1 per article (seed means)')
GAPS = {
    'objective (oracle − best on P*)':       (ART.oracle_r1 - ART.front_r1).to_numpy(),
    'search (best on P* − best in archive)': ART.front_r1.to_numpy() - sm(FULL, 'bestout_r1'),
    'selection (1,1)':                       sm(FULL, 'bestout_r1') - sm(FULL, 'r1_w1'),
    'selection (32,1)':                      sm(FULL, 'bestout_r1') - sm(FULL, 'r1_w32'),
}
out('gap\tmean\tSD\tmin\tQ1\tmedian\tQ3\tmax\tarticles with gap < 0.001')
for k, x in GAPS.items():
    q = np.percentile(x, [25, 50, 75])
    out(f'{k}\t{x.mean():.4f}\t{x.std(ddof=1):.4f}\t{x.min():.4f}\t{q[0]:.4f}\t{q[1]:.4f}\t{q[2]:.4f}\t'
        f'{x.max():.4f}\t{int((x < 1e-3).sum())}')
for j, m in ((1, 'r2'), (2, 'rl')):
    out(f'  {m.upper()}: oracle {ART[f"oracle_{m}"].mean():.4f} | best on P* {ART[f"front_{m}"].mean():.4f} | '
        f'best in archive {sm(FULL, f"bestout_{m}").mean():.4f} | delivered (1,1) {sm(FULL, f"{m}_w1").mean():.4f} | '
        f'(32,1) {sm(FULL, f"{m}_w32").mean():.4f}')

out('\nSWITCHING WEIGHT w* (Q22): for w_cov > w* the rule returns the output set\'s max-coverage member')
out('stage\tcov.extreme in output%\tw* median\tp90\tmax\truns w*>1\truns w*>=32\truns w*>=64')
for s in SNAMES:
    g = RUNS[RUNS.stage == s]
    out(f'{s}\t{100 * g.ext_term.mean():.1f}\t{g.wstar.median():.2f}\t{g.wstar.quantile(.9):.2f}\t'
        f'{g.wstar.max():.2f}\t{int((g.wstar > 1).sum())}\t{int((g.wstar >= 32).sum())}\t{int((g.wstar >= 64).sum())}')
g = RUNS[RUNS.stage == FULL]
hi = g[g.wstar >= 32].groupby('article').wstar.agg(['count', 'median', 'max'])
out('  +LS, articles (1-based) with w* >= 32 -> runs, median w*, max w*: ' +
    str({int(a) + 1: (int(r['count']), round(float(r['median']), 1), round(float(r['max']), 1)) for a, r in hi.iterrows()}))

A149 = 148 if 148 in enums else HARD[0]
e, R = enums[A149], SCORES[A149][0]
k_or, k_bf = int(np.argmax(R[:, 0])), int(np.where(PF[A149])[0][np.argmax(R[PF[A149], 0])])
out(f'\nARTIKEL {A149 + 1} (Q27): N {e.cache.n}, C(N,3) {e.M}, |P*| {e.m}')
for lab, row in (('oracle (R-1)', k_or), ('best on P* (R-1)', k_bf), ('LEAD-3', LEAD[A149])):
    out(f'  {lab:<18} sentences {[int(i) + 1 for i in e.combos[row]]}  cov {e.F_native[row, 0]:.6f}  '
        f'red {e.F_native[row, 1]:.6f}  R-1/2/L {R[row, 0]:.4f}/{R[row, 1]:.4f}/{R[row, 2]:.4f}  on P* {bool(PF[A149][row])}')
g = RUNS[(RUNS.stage == FULL) & (RUNS.article == A149)]
for wk in ('w1', 'w32'):
    cnt = collections.Counter(getattr(g, f'row_{wk}'))
    out(f'  +LS {wk}: ' + '; '.join(f'sentences {[int(i) + 1 for i in e.combos[row]]} in {c} seeds '
                                    f'(R-1 {R[row, 0]:.4f}, cov {e.F_native[row, 0]:.6f}, red {e.F_native[row, 1]:.6f})'
                                    for row, c in cnt.most_common()))

## C8 - figures (thesis in Indonesian with decimal comma, paper in English)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
INK, INK2, GRID, BG = '#0b0b0b', '#52514e', '#e6e5e0', '#c3c2b7'
C1, C2, C3 = '#2a78d6', '#eb6834', '#1baf7a'          # first 3 colours of the palette
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2,
                     'ytick.color': INK2, 'text.color': INK, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.dpi': 300, 'figure.dpi': 110})
L = {'id': dict(cov='coverage $f_1$ (dimaksimalkan)', red='redundancy $f_2$ (diminimalkan)',
               cand='seluruh {M} kandidat', pstar='Pareto front sejati P*', orc='oracle ROUGE-1',
               d1='terpilih (1, 1)', d32='terpilih (32, 1)', bf='terbaik di P*',
               gx='selisih ROUGE-1 F1 per artikel', g=['celah objektif\n(oracle − terbaik di P*)',
               'celah pencarian\n(terbaik di P* − terbaik di arsip)', 'celah seleksi, bobot (1, 1)',
               'celah seleksi, bobot (32, 1)'], hv='HV, % dari optimum\n(33 artikel sulit)',
               r1='ROUGE-1 F1\n(150 artikel)', w1='bobot (1, 1)', w32='bobot (32, 1)', lead='LEAD-3',
               stage='tahap ablasi', dec=','),
     'en': dict(cov='coverage $f_1$ (maximised)', red='redundancy $f_2$ (minimised)', cand='all {M} candidates',
               pstar='true Pareto front P*', orc='ROUGE-1 oracle', d1='selected (1, 1)',
               d32='selected (32, 1)', bf='best on P*', gx='ROUGE-1 F1 difference per article',
               g=['objective gap\n(oracle − best on P*)', 'search gap\n(best on P* − best in archive)',
                  'selection gap, weights (1, 1)', 'selection gap, weights (32, 1)'],
               hv='HV, % of optimum\n(33 hard articles)', r1='ROUGE-1 F1\n(150 articles)', w1='weights (1, 1)',
               w32='weights (32, 1)', lead='LEAD-3', stage='ablation stage', dec='.')}

def fmt(dec, nd):
    return FuncFormatter(lambda v, p: f'{v:.{nd}f}'.replace('.', dec))

def fig_article(lang):
    t = L[lang]; dec = t['dec']
    e, R = enums[A149], SCORES[A149][0]
    g = RUNS[(RUNS.stage == FULL) & (RUNS.article == A149) & (RUNS.seed == SEEDS[0])].iloc[0]
    fig, ax = plt.subplots(figsize=(6.3, 4.2))
    ax.scatter(e.F_native[:, 0], e.F_native[:, 1], s=5, c=BG, alpha=.6, lw=0,
               label=t['cand'].format(M=f'{e.M:,}'.replace(',', '.' if dec == ',' else ',')))
    P = e.P_star_min[np.argsort(e.P_star_min[:, 0])]
    ax.plot(-P[:, 0], P[:, 1], '-o', c=INK, ms=4, lw=1.2, label=t['pstar'], zorder=3)
    roles = [(int(g.row_w1), t['d1'], C1, 'o', 70), (int(g.row_w32), t['d32'], C2, 's', 70),
             (k_or, t['orc'], C3, '*', 160), (k_bf, t['bf'], INK, 'D', 55), (LEAD[A149], t['lead'], INK2, '^', 60)]
    groups = {}
    for row, lab, col, mk, sz in roles:                 # one marker per distinct summary, all roles in its label
        groups.setdefault(row, []).append((lab, col, mk, sz))
    for row, rl in groups.items():
        lab, col, mk, sz = rl[0]
        ax.scatter(*e.F_native[row], s=sz, c=col, marker=mk, edgecolor='white', lw=1.5, zorder=5,
                   label=' = '.join(x[0] for x in rl))
        ax.annotate(f'R-1 {R[row, 0]:.4f}'.replace('.', dec), e.F_native[row], xytext=(7, 4),
                    textcoords='offset points', fontsize=8, color=INK)
    ax.set_xlabel(t['cov']); ax.set_ylabel(t['red'])
    ax.xaxis.set_major_formatter(fmt(dec, 2)); ax.yaxis.set_major_formatter(fmt(dec, 2))
    ax.grid(color=GRID, lw=.5); ax.legend(fontsize=7.5, frameon=False, loc='best')
    fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fig_artikel{A149 + 1}_{lang}.png'); plt.close(fig)

def fig_gaps(lang):
    t = L[lang]; dec = t['dec']
    fig, ax = plt.subplots(figsize=(6.3, 3.4))
    rng = np.random.default_rng(0)
    for i, (k, x) in enumerate(GAPS.items()):
        y = len(GAPS) - 1 - i
        ax.scatter(x, y + rng.uniform(-.18, .18, len(x)), s=9, c=C1, alpha=.55, lw=0)
        q1, md, q3 = np.percentile(x, [25, 50, 75])
        ax.plot([q1, q3], [y, y], c=INK, lw=3, solid_capstyle='butt', alpha=.85)
        ax.plot([md, md], [y - .28, y + .28], c=INK, lw=1.5)
        ax.annotate(f'x̄ = {x.mean():.3f}'.replace('.', dec), (x.max(), y), xytext=(6, -3),
                    textcoords='offset points', fontsize=8, color=INK2)
    ax.set_yticks(range(len(GAPS))); ax.set_yticklabels(t['g'][::-1])
    ax.axvline(0, c=INK2, lw=.6); ax.set_xlabel(t['gx']); ax.xaxis.set_major_formatter(fmt(dec, 1))
    ax.grid(axis='x', color=GRID, lw=.5)
    fig.tight_layout(); fig.savefig(f'{FIG_DIR}/fig_gaps_{lang}.png'); plt.close(fig)

def fig_stages(lang):
    t = L[lang]; dec = t['dec']
    x = np.arange(len(SNAMES))
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(6.3, 4.6), sharex=True, layout='constrained')
    hv = [RUNS[(RUNS.stage == s) & RUNS.article.isin(HARD)].hv.mean() for s in SNAMES]
    a1.plot(x, hv, c=INK, marker='o', ms=5, lw=1.5)
    for xi, v in zip(x, hv):
        a1.annotate(f'{v:.2f}'.replace('.', dec), (xi, v), xytext=(0, 6), textcoords='offset points',
                    ha='center', fontsize=7.5, color=INK2)
    a1.set_ylabel(t['hv']); a1.yaxis.set_major_formatter(fmt(dec, 0 if max(hv) - min(hv) > 5 else 1)); a1.grid(axis='y', color=GRID, lw=.5)
    a1.margins(y=.25)
    ends = []
    for wk, col, lab, mk in (('w1', C1, t['w1'], 'o'), ('w32', C2, t['w32'], 's')):
        m = [sm(s, f'r1_{wk}').mean() for s in SNAMES]
        a2.plot(x, m, c=col, marker=mk, ms=5, lw=1.5, label=lab)
        ends.append([m[-1], lab])
    ends.sort()
    if ends[1][0] - ends[0][0] < .012:                  # keep the two end labels apart
        mid = (ends[0][0] + ends[1][0]) / 2
        ends[0][0], ends[1][0] = mid - .006, mid + .006
    for y, lab in ends:
        a2.annotate(lab, (x[-1], y), xytext=(8, -3), textcoords='offset points', fontsize=8, color=INK)
    a2.axhline(ART.lead3_r1.mean(), c=INK2, lw=1, ls='--')
    a2.annotate(t['lead'], (x[0], ART.lead3_r1.mean()), xytext=(0, 4), textcoords='offset points', fontsize=8, color=INK2)
    a2.set_ylabel(t['r1']); a2.yaxis.set_major_formatter(fmt(dec, 2)); a2.grid(axis='y', color=GRID, lw=.5)
    a2.legend(fontsize=7.5, frameon=False, loc='best')
    a2.set_xticks(x); a2.set_xticklabels(SNAMES); a2.set_xlabel(t['stage']); a2.set_xlim(-.4, len(SNAMES) - .1)
    fig.savefig(f'{FIG_DIR}/fig_stages_{lang}.png'); plt.close(fig)

for lang in ('id', 'en'):
    fig_article(lang); fig_gaps(lang); fig_stages(lang)
out(f'\n[C8] figures saved to {FIG_DIR}: {sorted(os.listdir(FIG_DIR))}')

## C9 - save

In [ ]:
PA = ART.copy()
for s in SNAMES:
    tag = s.replace('+', 'p').replace('Baseline', 'base')
    PA[f'{tag}_r1_w1'] = sm(s, 'r1_w1'); PA[f'{tag}_r1_w32'] = sm(s, 'r1_w32')
PA['full_bestout_r1'] = sm(FULL, 'bestout_r1')
PA['full_wstar_median'] = RUNS[RUNS.stage == FULL].groupby('article').wstar.median().reindex(AIDS).to_numpy()
PA = PA.round(4)
PA.index = PA.index + 1
PA.index.name = 'article'
PA.to_csv(f'{WP2_DIR}/wp2_per_article.csv')
keep = ['article', 'stage', 'seed', 'hv', 'recov', 'n_unique', 'first_gen', 'wstar', 'row_w1', 'row_w32',
        'r1_w1', 'r1_w32', 'bestout_r1', 'ext_term']
RUNS[keep].assign(article=RUNS.article + 1).round(5).to_csv(f'{WP2_DIR}/wp2_runs.csv', index=False)
with open(f'{WP2_DIR}/wp2_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(LOG))
for fn in sorted(os.listdir(WP2_DIR)):
    p = f'{WP2_DIR}/{fn}'
    if os.path.isfile(p):
        print(f'{fn:<26} {os.path.getsize(p) / 1024:8.1f} KB')
print('\nfiles to keep: wp2_summary.txt, wp2_per_article.csv '
      '(figures stay on Drive in wp2/figures/)')